### Mount Google Drive and initialise SparkSession.


In [ ]:
# Cell 1 — Mount Drive, initialise Spark, and import libraries
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import zipfile, pickle, json, time, gc, shutil, os
import numpy as np
import pandas as pd

from pyspark.sql import SparkSession, functions as F, types as T

spark = (
    SparkSession.builder
    .appName("7006SCN_Task1_WESAD")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

MYDRIVE = Path("/content/drive/MyDrive")
OUTPUT = MYDRIVE / "wesad_spark_parquet_fresh_run"
MANIFEST = OUTPUT / "_verification_manifest.json"

print("Spark version:", spark.version)
print("Drive root:", MYDRIVE)
print("Output folder:", OUTPUT)

Mounted at /content/drive
Spark version: 4.0.4
Drive root: /content/drive/MyDrive
Output folder: /content/drive/MyDrive/wesad_spark_parquet_fresh_run


### Locate and verify the original WESAD ZIP; record source file sizes.


In [ ]:
# Cell 2 — Locate and audit the original WESAD ZIP (read-only)
zip_candidates = [
    p for p in MYDRIVE.rglob("*")
    if p.is_file() and p.suffix.lower() == ".zip" and "wesad" in p.name.lower()
]
if not zip_candidates:
    raise FileNotFoundError("No WESAD ZIP found under MyDrive. No output has been deleted.")

preferred = [p for p in zip_candidates if p.name.lower() == "wesad (2).zip"]
zip_path = preferred[0] if preferred else max(zip_candidates, key=lambda p: p.stat().st_size)

print("Selected source ZIP:", zip_path)
print(f"ZIP size: {zip_path.stat().st_size:,} bytes ({zip_path.stat().st_size / (1024**3):.2f} GiB)")
print("\nAll matching ZIP files:")
for p in zip_candidates:
    print(f"  {p} | {p.stat().st_size:,} bytes")

with zipfile.ZipFile(zip_path, "r") as zf:
    bad_member = zf.testzip()
    if bad_member:
        raise RuntimeError(f"ZIP integrity check failed at member: {bad_member}")
    zip_members = zf.infolist()
    pkl_members = [m for m in zip_members if m.filename.lower().endswith(".pkl")]
    uncompressed_pkl_bytes = sum(m.file_size for m in pkl_members)

print("\nZIP integrity test: PASS")
print("Total ZIP members:", len(zip_members))
print("Pickle members:", len(pkl_members))
print(f"Total uncompressed pickle size: {uncompressed_pkl_bytes:,} bytes ({uncompressed_pkl_bytes / (1024**3):.2f} GiB)")

Selected source ZIP: /content/drive/MyDrive/WESAD (2).zip
ZIP size: 2,249,444,501 bytes (2.09 GiB)

All matching ZIP files:
  /content/drive/MyDrive/WESAD (2).zip | 2,249,444,501 bytes

ZIP integrity test: PASS
Total ZIP members: 92
Pickle members: 15
Total uncompressed pickle size: 13,827,624,985 bytes (12.88 GiB)


### Identify and verify the 15 expected participant pickle files.


In [ ]:
# Cell 3 — Identify the 15 expected WESAD participant pickles
EXPECTED_SUBJECTS = [
    "S2","S3","S4","S5","S6","S7","S8","S9","S10","S11",
    "S13","S14","S15","S16","S17"
]

with zipfile.ZipFile(zip_path, "r") as zf:
    all_names = zf.namelist()
    pkl_names = [n for n in all_names if n.lower().endswith(".pkl")]

subject_member = {}
for subject in EXPECTED_SUBJECTS:
    # Standard UCI WESAD archive contains participant files named Sx.pkl.
    matches = [n for n in pkl_names if Path(n).stem.lower() == subject.lower()]
    if len(matches) == 1:
        subject_member[subject] = matches[0]
    elif len(matches) > 1:
        raise RuntimeError(f"Multiple exact participant pickle files found for {subject}: {matches}")

print("Expected participant count:", len(EXPECTED_SUBJECTS))
print("Matched participant count:", len(subject_member))
for subject in EXPECTED_SUBJECTS:
    print(f"{subject}: {subject_member.get(subject, 'MISSING')}")

missing = [s for s in EXPECTED_SUBJECTS if s not in subject_member]
if missing:
    print("\nAll pickle filenames found in ZIP:")
    for n in pkl_names:
        print(" ", n)
    raise RuntimeError(f"Participant-file audit failed. Missing: {missing}. Do not start the build until paths are reconciled.")

Expected participant count: 15
Matched participant count: 15
S2: WESAD/S2/S2.pkl
S3: WESAD/S3/S3.pkl
S4: WESAD/S4/S4.pkl
S5: WESAD/S5/S5.pkl
S6: WESAD/S6/S6.pkl
S7: WESAD/S7/S7.pkl
S8: WESAD/S8/S8.pkl
S9: WESAD/S9/S9.pkl
S10: WESAD/S10/S10.pkl
S11: WESAD/S11/S11.pkl
S13: WESAD/S13/S13.pkl
S14: WESAD/S14/S14.pkl
S15: WESAD/S15/S15.pkl
S16: WESAD/S16/S16.pkl
S17: WESAD/S17/S17.pkl


### Define schema


In [ ]:
# Fresh output folder: never read or reuse previous Parquet outputs.
if OUTPUT.exists():
    raise FileExistsError(
        f"Fresh output folder already exists: {OUTPUT}. "
        "Stop and inspect/remove this exact generated folder before running a fresh build."
    )
OUTPUT.mkdir(parents=True, exist_ok=False)

FEATURES = ["ACC_X", "ACC_Y", "ACC_Z", "ECG", "EMG", "EDA", "Temp", "Resp"]
SCHEMA = T.StructType(
    [T.StructField(c, T.DoubleType(), True) for c in FEATURES] +
    [
        T.StructField("label", T.IntegerType(), False),
        T.StructField("subject_id", T.StringType(), False),
        T.StructField("sample_index", T.LongType(), False),
    ]
)
TEMP_SUFFIX = "_building"
completed = {}
manifest = {"source_archive": zip_path.name, "participants": {}}
MANIFEST.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Fresh output folder created:", OUTPUT)
print("Previous Parquet folders will not be read or reused.")


Fresh output folder created: /content/drive/MyDrive/wesad_spark_parquet_fresh_run
Previous Parquet folders will not be read or reused.


### Build all 15 participant Parquet outputs from the ZIP; validate each before completion.


In [ ]:
# Cell 5 — Build missing participants in chunks (fast path, but checkpointed)
# Increase CHUNK_ROWS only if Colab has sufficient RAM. No fixed runtime is guaranteed.
CHUNK_ROWS = 500_000

def extract_arrays(data, subject):
    if not isinstance(data, dict) or "signal" not in data or "label" not in data:
        raise ValueError(f"{subject}: pickle does not contain expected 'signal' and 'label' keys.")
    chest = data["signal"].get("chest")
    if chest is None:
        raise ValueError(f"{subject}: chest-worn signal group not found.")

    required = ["ACC", "ECG", "EMG", "EDA", "Temp", "Resp"]
    missing_channels = [k for k in required if k not in chest]
    if missing_channels:
        raise ValueError(f"{subject}: missing chest channels: {missing_channels}")

    acc = np.asarray(chest["ACC"])
    if acc.ndim != 2 or acc.shape[1] < 3:
        raise ValueError(f"{subject}: unexpected ACC shape {acc.shape}")

    arrays = {
        "ACC_X": np.asarray(acc[:, 0]).reshape(-1),
        "ACC_Y": np.asarray(acc[:, 1]).reshape(-1),
        "ACC_Z": np.asarray(acc[:, 2]).reshape(-1),
        "ECG": np.asarray(chest["ECG"]).reshape(-1),
        "EMG": np.asarray(chest["EMG"]).reshape(-1),
        "EDA": np.asarray(chest["EDA"]).reshape(-1),
        "Temp": np.asarray(chest["Temp"]).reshape(-1),
        "Resp": np.asarray(chest["Resp"]).reshape(-1),
    }
    labels = np.asarray(data["label"]).reshape(-1)
    lengths = {k: len(v) for k, v in arrays.items()}
    lengths["label"] = len(labels)
    if len(set(lengths.values())) != 1:
        raise ValueError(f"{subject}: signal/label length mismatch: {lengths}")
    return arrays, labels

with zipfile.ZipFile(zip_path, "r") as zf:
    for subject in EXPECTED_SUBJECTS:
        if subject in completed:
            raise RuntimeError(f"Unexpected pre-completed subject in fresh run: {subject}")

        final_dir = OUTPUT / subject
        temp_dir = OUTPUT / f".{subject}{TEMP_SUFFIX}"
        if temp_dir.exists():
            shutil.rmtree(temp_dir)
        if final_dir.exists():
            shutil.rmtree(final_dir)

        print(f"\nBuilding {subject} from {subject_member[subject]}")
        started = time.time()
        try:
            raw = zf.read(subject_member[subject])
            try:
                data = pickle.loads(raw, encoding="latin1")
            except TypeError:
                data = pickle.loads(raw)
            del raw

            arrays, labels = extract_arrays(data, subject)
            n = len(labels)
            if n == 0:
                raise ValueError(f"{subject}: zero samples.")

            # Create the temporary output only for this participant.
            temp_dir.mkdir(parents=True, exist_ok=True)
            for start in range(0, n, CHUNK_ROWS):
                end = min(start + CHUNK_ROWS, n)
                payload = {k: np.asarray(v[start:end], dtype=np.float64) for k, v in arrays.items()}
                payload["label"] = np.asarray(labels[start:end], dtype=np.int32)
                payload["subject_id"] = np.repeat(subject, end - start)
                payload["sample_index"] = np.arange(start, end, dtype=np.int64)

                pdf = pd.DataFrame(payload, columns=FEATURES + ["label", "subject_id", "sample_index"])
                sdf = spark.createDataFrame(pdf, schema=SCHEMA)
                sdf.write.mode("append").option("compression", "snappy").parquet(str(temp_dir))

                if start == 0 or end == n or (start // CHUNK_ROWS) % 4 == 0:
                    print(f"  wrote {end:,}/{n:,} rows")
                del payload, pdf, sdf
                gc.collect()

            # Reload the just-written participant and validate before promotion.
            check = spark.read.schema(SCHEMA).parquet(str(temp_dir))
            actual_rows = check.count()
            wrong_id = check.filter(F.col("subject_id") != subject).limit(1).count()
            null_labels = check.filter(F.col("label").isNull()).limit(1).count()
            if actual_rows != n or wrong_id > 0 or null_labels > 0:
                raise RuntimeError(
                    f"{subject} validation failed: expected={n:,}, actual={actual_rows:,}, "
                    f"wrong_subject={wrong_id}, null_labels={null_labels}"
                )

            # Promote only after all participant checks pass.
            temp_dir.rename(final_dir)
            completed[subject] = actual_rows
            manifest["participants"][subject] = {
                "rows": int(actual_rows),
                "status": "verified",
                "source_member": subject_member[subject],
                "elapsed_seconds": round(time.time() - started, 2),
            }
            MANIFEST.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
            print(f"VERIFIED {subject}: {actual_rows:,} rows in {(time.time()-started)/60:.1f} min")

            del data, arrays, labels, check
            gc.collect()

        except Exception as e:
            print(f"FAILED {subject}: {type(e).__name__}: {e}")
            if temp_dir.exists():
                shutil.rmtree(temp_dir)
            raise


Building S2 from WESAD/S2/S2.pkl
  wrote 500,000/4,255,300 rows
  wrote 2,500,000/4,255,300 rows
  wrote 4,255,300/4,255,300 rows
VERIFIED S2: 4,255,300 rows in 2.0 min

Building S3 from WESAD/S3/S3.pkl
  wrote 500,000/4,545,100 rows
  wrote 2,500,000/4,545,100 rows
  wrote 4,500,000/4,545,100 rows
  wrote 4,545,100/4,545,100 rows
VERIFIED S3: 4,545,100 rows in 1.7 min

Building S4 from WESAD/S4/S4.pkl
  wrote 500,000/4,496,100 rows
  wrote 2,500,000/4,496,100 rows
  wrote 4,496,100/4,496,100 rows
VERIFIED S4: 4,496,100 rows in 1.6 min

Building S5 from WESAD/S5/S5.pkl
  wrote 500,000/4,380,600 rows
  wrote 2,500,000/4,380,600 rows
  wrote 4,380,600/4,380,600 rows
VERIFIED S5: 4,380,600 rows in 1.5 min

Building S6 from WESAD/S6/S6.pkl
  wrote 500,000/4,949,700 rows
  wrote 2,500,000/4,949,700 rows
  wrote 4,500,000/4,949,700 rows
  wrote 4,949,700/4,949,700 rows
VERIFIED S6: 4,949,700 rows in 1.8 min

Building S7 from WESAD/S7/S7.pkl
  wrote 500,000/3,666,600 rows
  wrote 2,500,000/3

### Verify all 15 participants, combined rows, schema, sample records and file sizes.


In [ ]:
from pathlib import Path

def parquet_files(folder):
    folder = Path(folder)
    return list(folder.rglob("*.parquet")) if folder.exists() else []

print("Parquet helper defined. Ready to rerun final verification.")

Parquet helper defined. Ready to rerun final verification.


In [ ]:
# Cell 6 — Final Task 1 verification: every participant, row count, schema, and file size
participant_counts = {}
for subject in EXPECTED_SUBJECTS:
    folder = OUTPUT / subject
    if not folder.exists() or not parquet_files(folder):
        raise RuntimeError(f"Final verification failed: no completed Parquet data for {subject}.")

    sdf = spark.read.schema(SCHEMA).parquet(str(folder))
    rows = sdf.count()
    if rows <= 0:
        raise RuntimeError(f"Final verification failed: {subject} has no rows.")
    if sdf.filter(F.col("subject_id") != subject).limit(1).count() > 0:
        raise RuntimeError(f"Final verification failed: subject_id mismatch for {subject}.")
    participant_counts[subject] = rows

df = spark.read.schema(SCHEMA).parquet(*[str(OUTPUT / s) for s in EXPECTED_SUBJECTS])
total_rows = df.count()
total_columns = len(df.columns)
expected_rows = sum(participant_counts.values())
parquet_bytes = sum(p.stat().st_size for p in OUTPUT.rglob("*.parquet") if p.is_file())

print("PARTICIPANT ROW COUNTS")
for subject in EXPECTED_SUBJECTS:
    print(f"{subject}: {participant_counts[subject]:,}")

print("\nPARTICIPANTS VERIFIED:", len(participant_counts), "/ 15")
print(f"COMBINED ROWS: {total_rows:,}")
print("COLUMNS:", total_columns, df.columns)
print("ROWS MATCH PARTICIPANT SUM:", total_rows == expected_rows)
print(f"ORIGINAL ZIP SIZE: {zip_path.stat().st_size:,} bytes ({zip_path.stat().st_size / (1024**3):.2f} GiB)")
print(f"UNCOMPRESSED PICKLE MEMBERS: {uncompressed_pkl_bytes:,} bytes ({uncompressed_pkl_bytes / (1024**3):.2f} GiB)")
print(f"GENERATED PARQUET SIZE: {parquet_bytes:,} bytes ({parquet_bytes / (1024**3):.2f} GiB)")
print("10M ROW REQUIREMENT:", "PASS" if total_rows >= 10_000_000 else "FAIL")
print("10 COLUMN REQUIREMENT:", "PASS" if total_columns >= 10 else "FAIL")

df.printSchema()
df.show(20, truncate=False)

if len(participant_counts) != 15:
    raise RuntimeError("FAIL: expected all 15 participants.")
if total_rows != expected_rows:
    raise RuntimeError("FAIL: combined row count differs from participant row sum.")
if total_rows < 10_000_000 or total_columns < 10:
    raise RuntimeError("FAIL: coursework dataset size thresholds not met.")

final_manifest = {
    "source_archive": zip_path.name,
    "participant_count": len(participant_counts),
    "participant_rows": participant_counts,
    "total_rows": int(total_rows),
    "total_columns": int(total_columns),
    "columns": df.columns,
    "zip_size_bytes": int(zip_path.stat().st_size),
    "uncompressed_pickle_bytes": int(uncompressed_pkl_bytes),
    "parquet_size_bytes": int(parquet_bytes),
    "status": "verified"
}
MANIFEST.write_text(json.dumps(final_manifest, indent=2), encoding="utf-8")

print("\nTASK 1 FINAL GATE: PASS — all 15 participants and dataset checks verified.")

PARTICIPANT ROW COUNTS
S2: 4,255,300
S3: 4,545,100
S4: 4,496,100
S5: 4,380,600
S6: 4,949,700
S7: 3,666,600
S8: 3,826,200
S9: 3,656,100
S10: 3,847,200
S11: 3,663,100
S13: 3,875,900
S14: 3,883,600
S15: 3,676,400
S16: 3,941,700
S17: 4,144,000

PARTICIPANTS VERIFIED: 15 / 15
COMBINED ROWS: 60,807,600
COLUMNS: 11 ['ACC_X', 'ACC_Y', 'ACC_Z', 'ECG', 'EMG', 'EDA', 'Temp', 'Resp', 'label', 'subject_id', 'sample_index']
ROWS MATCH PARTICIPANT SUM: True
ORIGINAL ZIP SIZE: 2,249,444,501 bytes (2.09 GiB)
UNCOMPRESSED PICKLE MEMBERS: 13,827,624,985 bytes (12.88 GiB)
GENERATED PARQUET SIZE: 1,004,704,516 bytes (0.94 GiB)
10M ROW REQUIREMENT: PASS
10 COLUMN REQUIREMENT: PASS
root
 |-- ACC_X: double (nullable = true)
 |-- ACC_Y: double (nullable = true)
 |-- ACC_Z: double (nullable = true)
 |-- ECG: double (nullable = true)
 |-- EMG: double (nullable = true)
 |-- EDA: double (nullable = true)
 |-- Temp: double (nullable = true)
 |-- Resp: double (nullable = true)
 |-- label: integer (nullable = true)
 